In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from CRUD_Python_Module2 import AnimalShelter



###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "aacuserPa55w0rD"
shelter = AnimalShelter(username, password)

# Overload shelter to handle host, port, database and collection:
shelter = AnimalShelter(username, password, host="localhost", port=27017, db="aac", col="animals")

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('SimpleExample')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
    html.Hr(),
    
    # DataTable interactive module
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        
        # Page size limits the rows per page:
        page_size=10,
        # Enable sorting:
        sort_action='native',
        # Enable filtering:
        filter_action='native',
        # Enable single row selection:
        row_selectable='single',
        # Set default to first row:
        selected_rows=[0],
        
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px', 'width': '150px', 'maxWidth': '180px'},


    ),
    html.Br(),
    html.Hr(),
    html.Div(
            id='map-id',
            className='col s12 m6'),
    # Unique identifier:
    html.H3("Dashboard created by Joseph Glista"),
    html.Div("JSG Dashboard v1.1", style={"color": "green", "fontWeight": "bold"})
])


#############################################
# Interaction Between Components / Controller
#############################################
#This callback will highlight a row on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # only loop if selected_columns is not None
    if not selected_columns:
        return []
    # if it's not None, then return the data table
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):    
    dff = pd.DataFrame.from_dict(viewData)
 # Because we only allow single row selection, the list can 
 # be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Ensure there are enough columns first:
    if dff.shape[1] <= 14:
        return [dl.Map(style={'width': '1000px', 'height': '500px'},
               # Austin TX is at [30.75,-97.48]
            center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id")]
                      )
               ]
    
    # Normal case
       # Marker with tool tip and popup
       # Column 13 and 14 define the grid-coordinates for 
       # the map
       # Column 4 defines the breed for the animal
       # Column 9 defines the name of the animal
    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48], zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),        
                dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]],
                    children=[
                    dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9]),
                    html.Div(id='map-id', className='col s12 m6')
                    ])
                ])
            ])
    ]


    
# Run app and display result in jupyterlab mode.
########## if port 8050 is busy, try port=8051. ##########
app.run_server(mode='inline', port=8050)